# Time-Series Single-Contract Factor Research

This notebook tests factors on **individual futures contracts** (time-series approach).
Unlike the cross-sectional `research_workflow_futures.ipynb`, each model here trains
on a **single contract's history** and predicts that contract's own future return.

**Workflow:**
1. Liquidity screening — rank all contracts by avg daily turnover, keep Top N
2. IC analysis — for each (contract, factor, horizon 1..10d): compute Spearman IC and ICIR
3. IC heatmaps — visualise which factors work on which contracts at which horizons
4. ML per contract — train LightGBM per contract, evaluate OOS Sharpe
5. Summary report — cross-contract comparison table

---

## Factor Reference: Alpha158 (158 factors)

All 158 factors are derived from daily OHLCV data. Windows `W ∈ {5, 10, 20, 30, 60}` apply to all time-series families.

### Group 1 — Candlestick Patterns (9 factors)

| Factor | Formula | What it measures |
|--------|---------|-----------------|
| `kmid` | `(close - open) / open` | Body size relative to open — positive = bullish candle |
| `klen` | `(high - low) / open` | Total candle range — proxy for intraday volatility |
| `kmid_2` | `(close - open) / (high - low)` | Body as fraction of total range — candlestick fill ratio |
| `kup` | `(high - max(open,close)) / open` | Upper shadow — selling pressure after rally |
| `kup_2` | `(high - max(open,close)) / (high - low)` | Upper shadow as fraction of range |
| `klow` | `(min(open,close) - low) / open` | Lower shadow — buying support after drop |
| `klow_2` | `(min(open,close) - low) / (high - low)` | Lower shadow as fraction of range |
| `ksft` | `(2×close - high - low) / open` | Close position within range relative to open |
| `ksft_2` | `(2×close - high - low) / (high - low)` | Close position within range (0=bottom, 1=top) |

### Group 2 — Price Ratios (4 factors)

| Factor | Formula | What it measures |
|--------|---------|-----------------|
| `open_0` | `open / close` | Gap between open and close — overnight drift |
| `high_0` | `high / close` | How far today's high is above close — intraday spike |
| `low_0` | `low / close` | How far today's low is below close — intraday dip |
| `vwap_0` | `vwap / close` | Whether close is above/below volume-weighted average price |

### Group 3 — Price Time-Series (7 families × 5 windows = 35 factors)

| Family | Formula | What it measures |
|--------|---------|-----------------|
| `roc_W` | `ts_delay(close, W) / close` | Rate of change: today's price relative to W days ago (momentum) |
| `ma_W` | `ts_mean(close, W) / close` | W-day moving average relative to today — mean reversion signal |
| `std_W` | `ts_std(close, W) / close` | W-day price volatility normalised by price level |
| `beta_W` | `ts_slope(close, W) / close` | Linear trend slope of price over W days |
| `rsqr_W` | `ts_rsquare(close, W)` | R² of linear fit — how "trend-like" vs noisy the price path is |
| `resi_W` | `ts_resi(close, W) / close` | Residual from linear trend — mean-reversion component |
| `max_W` | `ts_max(high, W) / close` | W-day high relative to today's close — distance from peak |
| `min_W` | `ts_min(low, W) / close` | W-day low relative to today's close — distance from trough |

### Group 4 — Price Quantile & Rank (5 families × 5 windows = 25 factors)

| Family | Formula | What it measures |
|--------|---------|-----------------|
| `qtlu_W` | `ts_quantile(close, W, 0.8) / close` | 80th percentile of close over W days vs today — near top of range? |
| `qtld_W` | `ts_quantile(close, W, 0.2) / close` | 20th percentile vs today — near bottom of range? |
| `rank_W` | `ts_rank(close, W)` | Percentile rank of today's close in the last W days (0–1) |
| `rsv_W` | `(close - W-day low) / (W-day high - W-day low)` | Stochastic %K — where close sits in the W-day price range |
| `imax_W` | `ts_argmax(high, W) / W` | How many days ago was the W-day high (0=today, 1=W days ago) |
| `imin_W` | `ts_argmin(low, W) / W` | How many days ago was the W-day low |
| `imxd_W` | `(argmax - argmin) / W` | Temporal gap between W-day high and low — trend structure |

### Group 5 — Price-Volume Correlation (2 families × 5 windows = 10 factors)

| Family | Formula | What it measures |
|--------|---------|-----------------|
| `corr_W` | `ts_corr(close, log(volume+1), W)` | Correlation between price level and volume — "smart money" alignment |
| `cord_W` | `ts_corr(Δclose%, Δlog(volume%), W)` | Correlation between price change and volume change — follow-through |

### Group 6 — Up/Down Day Counts (3 families × 5 windows = 15 factors)

| Family | Formula | What it measures |
|--------|---------|-----------------|
| `cntp_W` | fraction of up days in W | Proportion of up-closes in window — trend persistence |
| `cntn_W` | fraction of down days in W | Proportion of down-closes |
| `cntd_W` | `cntp - cntn` | Net up-day dominance in window |

### Group 7 — Up/Down Move Size (3 families × 5 windows = 15 factors)

| Family | Formula | What it measures |
|--------|---------|-----------------|
| `sump_W` | Σ(up moves) / Σ(|all moves|) | Share of absolute movement that was upward (like RSI numerator) |
| `sumn_W` | Σ(down moves) / Σ(|all moves|) | Share of absolute movement that was downward |
| `sumd_W` | `sump - sumn` | Net directional bias in move magnitudes |

### Group 8 — Volume Time-Series (5 families × 5 windows = 25 factors)

| Family | Formula | What it measures |
|--------|---------|-----------------|
| `vma_W` | `ts_mean(volume, W) / volume` | Today's volume relative to W-day average — above/below normal |
| `vstd_W` | `ts_std(volume, W) / volume` | W-day volume volatility relative to today — volume regime |
| `wvma_W` | CoV of (return × volume) over W | Coefficient of variation of dollar-volume-weighted returns — liquidity-adjusted vol |
| `vsump_W` | fraction of volume-increase days | Proportion of days where volume was higher than previous day |
| `vsumn_W` | fraction of volume-decrease days | Proportion of days where volume was lower |
| `vsumd_W` | `vsump - vsumn` | Net volume expansion/contraction bias |

---

## Information Coefficient (IC) — How It Is Computed

IC measures how well a factor **predicts future returns** over a given horizon. Here we use the **time-series IC** variant: for a single contract, we test whether today's factor value predicts the contract's own return N days later.

### Formula

For a factor `f` and forward horizon `h`:

```
IC(f, h) = Spearman rank correlation( f[t], r[t, t+h] )
         = corr( rank(f[t]), rank(r[t, t+h]) )

where:
  f[t]       = factor value on day t
  r[t, t+h]  = h-day forward return = close[t+h] / close[t] - 1
```

We use **Spearman** (rank-based) rather than Pearson correlation because:
- Factor values are often non-normal (heavy tails, skewed)
- Rank correlation is robust to outliers
- It captures monotonic relationships, not just linear ones

### From IC to ICIR

A single IC value is noisy. We compute IC across all training days `t`, then summarise:

```
Mean IC  = mean over t of IC(f, h, t)    — average predictive strength
Std IC   = std over t of IC(f, h, t)     — stability of the signal
ICIR     = Mean IC / Std IC              — signal-to-noise ratio (like Sharpe)
```

**Interpretation guidelines:**

| Metric | Threshold | Meaning |
|--------|-----------|---------|
| `|Mean IC|` | > 0.05 | Factor has meaningful predictive content |
| `|Mean IC|` | > 0.10 | Strong factor — top-tier in practice |
| `|ICIR|` | > 0.5 | Signal is consistent (not just lucky) |
| `|ICIR|` | > 1.0 | Very consistent — robust across market regimes |

### What the sign means

- **IC > 0**: factor is positively predictive — high factor value → high future return (momentum)
- **IC < 0**: factor is negatively predictive — high factor value → low future return (contrarian/mean-reversion)
- Both are usable: a negative IC just means trade opposite to the factor direction

### Look-ahead bias precaution

Forward returns `r[t, t+h]` require future prices. To avoid leakage:
- IC is computed only on the **training period** (2019–2021)
- The **test period** (2023–2024) is used only for ML OOS evaluation, never for IC measurement
- In code: `dataset.fetch_raw(Segment.TRAIN)` is used, not the full dataset

In [ ]:
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy.stats import spearmanr

from vnpy.alpha import AlphaLab
from vnpy.alpha.dataset import AlphaDataset, Segment
from vnpy.alpha.model.models.lgb_model import LgbModel
from vnpy.trader.constant import Interval

# ─────────────────────────────────────────────
#  USER CONFIGURATION
# ─────────────────────────────────────────────

LAB_PATH     = "./lab/futures"
INTERVAL     = Interval.DAILY

TOP_N        = 15          # keep top N contracts by avg daily turnover
HORIZONS     = list(range(1, 11))   # 1 .. 10 day forward returns for IC analysis
ML_HORIZON   = 5           # forward return horizon used for ML label
EXTENDED_DAYS = 120        # extra lookback days for feature warm-up

TRAIN_PERIOD = ("2019-01-01", "2021-12-31")
VALID_PERIOD = ("2022-01-01", "2022-12-31")
TEST_PERIOD  = ("2023-01-01", "2024-12-31")

# ── Build all 158 Alpha158 factors programmatically ──────────────────
_windows = [5, 10, 20, 30, 60]

FACTORS = {}

# Candlestick pattern (9)
FACTORS["kmid"]   = "(close - open) / open"
FACTORS["klen"]   = "(high - low) / open"
FACTORS["kmid_2"] = "(close - open) / (high - low + 1e-12)"
FACTORS["kup"]    = "(high - ts_greater(open, close)) / open"
FACTORS["kup_2"]  = "(high - ts_greater(open, close)) / (high - low + 1e-12)"
FACTORS["klow"]   = "(ts_less(open, close) - low) / open"
FACTORS["klow_2"] = "((ts_less(open, close) - low) / (high - low + 1e-12))"
FACTORS["ksft"]   = "(close * 2 - high - low) / open"
FACTORS["ksft_2"] = "(close * 2 - high - low) / (high - low + 1e-12)"

# Price ratios (4)
for field in ["open", "high", "low", "vwap"]:
    FACTORS[f"{field}_0"] = f"{field} / close"

# Time-series price/volume features (5 windows × 27 families = 135)
for w in _windows:
    FACTORS[f"roc_{w}"]   = f"ts_delay(close, {w}) / close"
for w in _windows:
    FACTORS[f"ma_{w}"]    = f"ts_mean(close, {w}) / close"
for w in _windows:
    FACTORS[f"std_{w}"]   = f"ts_std(close, {w}) / close"
for w in _windows:
    FACTORS[f"beta_{w}"]  = f"ts_slope(close, {w}) / close"
for w in _windows:
    FACTORS[f"rsqr_{w}"]  = f"ts_rsquare(close, {w})"
for w in _windows:
    FACTORS[f"resi_{w}"]  = f"ts_resi(close, {w}) / close"
for w in _windows:
    FACTORS[f"max_{w}"]   = f"ts_max(high, {w}) / close"
for w in _windows:
    FACTORS[f"min_{w}"]   = f"ts_min(low, {w}) / close"
for w in _windows:
    FACTORS[f"qtlu_{w}"]  = f"ts_quantile(close, {w}, 0.8) / close"
for w in _windows:
    FACTORS[f"qtld_{w}"]  = f"ts_quantile(close, {w}, 0.2) / close"
for w in _windows:
    FACTORS[f"rank_{w}"]  = f"ts_rank(close, {w})"
for w in _windows:
    FACTORS[f"rsv_{w}"]   = f"(close - ts_min(low, {w})) / (ts_max(high, {w}) - ts_min(low, {w}) + 1e-12)"
for w in _windows:
    FACTORS[f"imax_{w}"]  = f"ts_argmax(high, {w}) / {w}"
for w in _windows:
    FACTORS[f"imin_{w}"]  = f"ts_argmin(low, {w}) / {w}"
for w in _windows:
    FACTORS[f"imxd_{w}"]  = f"(ts_argmax(high, {w}) - ts_argmin(low, {w})) / {w}"
for w in _windows:
    FACTORS[f"corr_{w}"]  = f"ts_corr(close, ts_log(volume + 1), {w})"
for w in _windows:
    FACTORS[f"cord_{w}"]  = f"ts_corr(close / ts_delay(close, 1), ts_log(volume / ts_delay(volume, 1) + 1), {w})"
for w in _windows:
    FACTORS[f"cntp_{w}"]  = f"ts_mean(close > ts_delay(close, 1), {w})"
for w in _windows:
    FACTORS[f"cntn_{w}"]  = f"ts_mean(close < ts_delay(close, 1), {w})"
for w in _windows:
    FACTORS[f"cntd_{w}"]  = f"ts_mean(close > ts_delay(close, 1), {w}) - ts_mean(close < ts_delay(close, 1), {w})"
for w in _windows:
    FACTORS[f"sump_{w}"]  = f"ts_sum(ts_greater(close - ts_delay(close, 1), 0), {w}) / (ts_sum(ts_abs(close - ts_delay(close, 1)), {w}) + 1e-12)"
for w in _windows:
    FACTORS[f"sumn_{w}"]  = f"ts_sum(ts_greater(ts_delay(close, 1) - close, 0), {w}) / (ts_sum(ts_abs(close - ts_delay(close, 1)), {w}) + 1e-12)"
for w in _windows:
    FACTORS[f"sumd_{w}"]  = f"(ts_sum(ts_greater(close - ts_delay(close, 1), 0), {w}) - ts_sum(ts_greater(ts_delay(close, 1) - close, 0), {w})) / (ts_sum(ts_abs(close - ts_delay(close, 1)), {w}) + 1e-12)"
for w in _windows:
    FACTORS[f"vma_{w}"]   = f"ts_mean(volume, {w}) / (volume + 1e-12)"
for w in _windows:
    FACTORS[f"vstd_{w}"]  = f"ts_std(volume, {w}) / (volume + 1e-12)"
for w in _windows:
    FACTORS[f"wvma_{w}"]  = f"ts_std(ts_abs(close / ts_delay(close, 1) - 1) * volume, {w}) / (ts_mean(ts_abs(close / ts_delay(close, 1) - 1) * volume, {w}) + 1e-12)"
for w in _windows:
    FACTORS[f"vsump_{w}"] = f"ts_sum(ts_greater(volume - ts_delay(volume, 1), 0), {w}) / (ts_sum(ts_abs(volume - ts_delay(volume, 1)), {w}) + 1e-12)"
for w in _windows:
    FACTORS[f"vsumn_{w}"] = f"ts_sum(ts_greater(ts_delay(volume, 1) - volume, 0), {w}) / (ts_sum(ts_abs(volume - ts_delay(volume, 1)), {w}) + 1e-12)"
for w in _windows:
    FACTORS[f"vsumd_{w}"] = f"(ts_sum(ts_greater(volume - ts_delay(volume, 1), 0), {w}) - ts_sum(ts_greater(ts_delay(volume, 1) - volume, 0), {w})) / (ts_sum(ts_abs(volume - ts_delay(volume, 1)), {w}) + 1e-12)"

# All futures candidates (same list as research_workflow_futures.ipynb)
ALL_FUTURES = [
    "rb88.SHFE", "hc88.SHFE", "i88.DCE", "j88.DCE", "jm88.DCE",
    "cu88.SHFE", "al88.SHFE", "zn88.SHFE", "ni88.SHFE",
    "au88.SHFE", "ag88.SHFE",
    "ru88.SHFE", "sc88.INE", "TA88.CZCE", "pp88.DCE",
    "m88.DCE", "y88.DCE", "p88.DCE", "c88.DCE", "a88.DCE",
    "SR88.CZCE", "CF88.CZCE",
]

lab = AlphaLab(LAB_PATH)
print(f"Config: TOP_N={TOP_N}, HORIZONS=1..{max(HORIZONS)}, ML_HORIZON={ML_HORIZON}")
print(f"Total factors: {len(FACTORS)}")
print(f"Factor families: kmid/klen/kup/klow/ksft(9), price_ratios(4), " +
      f"roc/ma/std/beta/rsqr/resi/max/min/qtlu/qtld/rank/rsv/imax/imin/imxd/" +
      f"corr/cord/cntp/cntn/cntd/sump/sumn/sumd/vma/vstd/wvma/vsump/vsumn/vsumd × {_windows}")

## Cell 2 — Liquidity Screening
Rank all contracts by average daily turnover (CNY) over the test period.
Turnover is more comparable than volume (which is in lots and varies by contract size).

In [ ]:
print("Loading turnover data for liquidity screening...")
df_liq = lab.load_bar_df(ALL_FUTURES, INTERVAL, TEST_PERIOD[0], TEST_PERIOD[1], extended_days=0)

# Note: load_bar_df normalises prices by dividing by first close, but volume/turnover are raw
liquidity = (
    df_liq
    .group_by("vt_symbol")
    .agg(
        pl.mean("turnover").alias("avg_daily_turnover"),
        pl.mean("volume").alias("avg_daily_volume"),
        pl.count("datetime").alias("trading_days")
    )
    .sort("avg_daily_turnover", descending=True)
)

# Display full ranking
liq_pd = liquidity.to_pandas()
liq_pd["rank"] = range(1, len(liq_pd) + 1)
liq_pd["avg_daily_turnover"] = (liq_pd["avg_daily_turnover"] / 1e8).round(2)  # in 亿元
liq_pd["avg_daily_volume"] = liq_pd["avg_daily_volume"].round(0).astype(int)
liq_pd = liq_pd[["rank", "vt_symbol", "avg_daily_turnover", "avg_daily_volume", "trading_days"]]
liq_pd.columns = ["Rank", "Contract", "Avg Daily Turnover (亿)", "Avg Daily Volume (lots)", "Trading Days"]
display(liq_pd)

SELECTED = liquidity.head(TOP_N)["vt_symbol"].to_list()
print(f"\nSelected top {TOP_N} by turnover:")
for i, s in enumerate(SELECTED, 1):
    print(f"  {i:2d}. {s}")

## Cell 3 — Load Full Dataset
Load OHLCV data for selected contracts covering train+valid+test periods plus warm-up days.

In [ ]:
print(f"Loading data for {len(SELECTED)} contracts ({TRAIN_PERIOD[0]} → {TEST_PERIOD[1]}, +{EXTENDED_DAYS}d warmup)...")
df = lab.load_bar_df(
    SELECTED, INTERVAL,
    start=TRAIN_PERIOD[0], end=TEST_PERIOD[1],
    extended_days=EXTENDED_DAYS
)

print(f"Loaded: {df.shape[0]:,} rows, {df['vt_symbol'].n_unique()} contracts")
print(f"Date range: {df['datetime'].min()} → {df['datetime'].max()}")
print(f"Columns: {df.columns}")

# Show per-contract row count
counts = df.group_by("vt_symbol").agg(pl.count("datetime").alias("bars")).sort("bars", descending=True)
display(counts.to_pandas())

## Cell 4 — IC Analysis (factors × horizons × contracts)

For each contract, compute Spearman IC between each factor and each forward return horizon.

**IC** = rank correlation between factor value today and N-day return starting tomorrow.  
**ICIR** = mean(IC) / std(IC) — signal consistency (analogous to Sharpe).  
Rule of thumb: |mean IC| > 0.05 and ICIR > 0.5 suggests a usable factor.

> Note: forward returns use `ts_delay(close, -h)/close - 1` which is the h-day return from today's close.

In [ ]:
from tqdm import tqdm as tqdm_nb

ic_records = []

for symbol in tqdm_nb(SELECTED, desc="Contracts"):
    df_c = df.filter(pl.col("vt_symbol") == symbol)

    dataset = AlphaDataset(df_c, TRAIN_PERIOD, VALID_PERIOD, TEST_PERIOD)

    # Add factors
    for name, expr in FACTORS.items():
        dataset.add_feature(name, expr)

    # Add forward returns for all horizons (used as IC targets, not ML labels)
    for h in HORIZONS:
        dataset.add_feature(f"fwd_{h}d", f"ts_delay(close, -{h}) / close - 1")

    # Use dummy label to satisfy API (will use fwd columns directly)
    dataset.set_label(f"ts_delay(close, -{ML_HORIZON}) / close - 1")
    dataset.prepare_data(max_workers=1)

    # Use train period for IC (avoid data leakage)
    raw = dataset.fetch_raw(Segment.TRAIN).to_pandas()
    raw = raw.dropna()

    if len(raw) < 60:
        print(f"  WARN: {symbol} has only {len(raw)} clean rows in train -- skipping")
        continue

    for factor_name in FACTORS:
        for h in HORIZONS:
            fwd_col = f"fwd_{h}d"
            if factor_name not in raw.columns or fwd_col not in raw.columns:
                continue
            sub = raw[[factor_name, fwd_col]].dropna()
            if len(sub) < 30:
                continue
            ic_val, _ = spearmanr(sub[factor_name], sub[fwd_col])
            ic_records.append({
                "symbol": symbol,
                "factor": factor_name,
                "horizon": h,
                "IC": round(ic_val, 4),
                "n": len(sub)
            })

ic_df = pd.DataFrame(ic_records)
print(f"\nIC records: {len(ic_df)}")

# Summary statistics
ic_summary = (
    ic_df.groupby(["factor", "horizon"])["IC"]
    .agg(["mean", "std", lambda x: (x.mean() / x.std()) if x.std() > 0 else 0])
    .rename(columns={"mean": "Mean_IC", "std": "Std_IC", "<lambda_0>": "ICIR"})
    .round(4)
)
print("\nIC Summary (mean across contracts):")
display(ic_summary)

## Cell 5 — IC Heatmaps
**Chart A**: Per-factor heatmap — rows=contracts, cols=horizons, color=IC  
**Chart B**: Cross-factor summary — rows=factors, cols=horizons, color=mean IC across contracts

In [ ]:
factor_names = list(FACTORS.keys())
n_factors = len(factor_names)

# ── Chart A: Cross-factor summary — rows=factors, cols=horizons ──────
# With 158 factors, show mean IC across contracts per (factor, horizon)
summary_pivot = (
    ic_df.groupby(["factor", "horizon"])["IC"]
    .mean()
    .reset_index()
    .pivot(index="factor", columns="horizon", values="IC")
    .reindex(columns=HORIZONS)
    .reindex(factor_names)
)

fig, ax = plt.subplots(figsize=(14, max(8, n_factors * 0.18 + 2)))
sns.heatmap(
    summary_pivot, ax=ax, center=0, cmap="RdYlGn",
    linewidths=0,
    vmin=-0.15, vmax=0.15,
    xticklabels=HORIZONS,
    yticklabels=True,
    cbar_kws={"label": "Mean IC across contracts"}
)
ax.set_title("Mean IC Across Contracts (rows=factors, cols=horizons 1-10d)",
             fontsize=12, fontweight="bold")
ax.set_xlabel("Forward Horizon (days)")
ax.set_ylabel("Factor")
ax.tick_params(axis='y', labelsize=7)
plt.tight_layout()
plt.savefig("./lab/futures/ts_ic_summary.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: ts_ic_summary.png")

# ── Chart B: Top/bottom 20 factors by |mean IC| at best horizon ──────
best_ic_per_factor = (
    ic_df.groupby(["factor", "horizon"])["IC"]
    .mean()
    .reset_index()
    .assign(abs_IC=lambda x: x["IC"].abs())
    .sort_values("abs_IC", ascending=False)
    .groupby("factor")
    .first()
    .reset_index()
    .sort_values("abs_IC", ascending=False)
)

top20 = best_ic_per_factor.head(20)
bot20 = best_ic_per_factor.tail(20)

fig2, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))
colors_top = ["#2ecc71" if v > 0 else "#e74c3c" for v in top20["IC"]]
ax1.barh(top20["factor"], top20["IC"], color=colors_top)
ax1.axvline(0, color="black", linewidth=0.8)
ax1.axvline(0.05, color="#27ae60", linewidth=1, linestyle=":", label="|IC|=0.05")
ax1.axvline(-0.05, color="#27ae60", linewidth=1, linestyle=":")
ax1.set_title("Top 20 Factors by |Mean IC| (best horizon)", fontweight="bold")
ax1.set_xlabel("Mean IC at best horizon")
ax1.legend()

colors_bot = ["#2ecc71" if v > 0 else "#e74c3c" for v in bot20["IC"]]
ax2.barh(bot20["factor"], bot20["IC"], color=colors_bot)
ax2.axvline(0, color="black", linewidth=0.8)
ax2.set_title("Bottom 20 Factors (weakest IC)", fontweight="bold")
ax2.set_xlabel("Mean IC at best horizon")

plt.tight_layout()
plt.savefig("./lab/futures/ts_ic_top_bottom.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: ts_ic_top_bottom.png")

# ── ICIR table — top 20 ───────────────────────────────────────────────
icir_per_factor = (
    ic_df.groupby(["factor", "horizon"])["IC"]
    .agg(lambda x: x.mean() / x.std() if x.std() > 0 else 0)
    .reset_index()
    .assign(abs_ICIR=lambda x: x["IC"].abs())
    .sort_values("abs_ICIR", ascending=False)
    .groupby("factor")
    .first()
    .reset_index()
    .rename(columns={"IC": "ICIR", "horizon": "best_horizon"})
    .sort_values("abs_ICIR", ascending=False)
)

print(f"\nTop 20 factors by |ICIR| (best horizon):")
display(icir_per_factor[["factor", "best_horizon", "ICIR"]].head(20).reset_index(drop=True))

# ── IC summary pivot (mean IC, top 20 factors × all horizons) ─────────
top20_factors = icir_per_factor.head(20)["factor"].tolist()
icir_pivot = (
    ic_df[ic_df["factor"].isin(top20_factors)]
    .groupby(["factor", "horizon"])["IC"]
    .agg(lambda x: x.mean() / x.std() if x.std() > 0 else 0)
    .reset_index()
    .pivot(index="factor", columns="horizon", values="IC")
    .reindex(columns=HORIZONS)
    .reindex(top20_factors)
    .round(2)
)
print(f"\nICIR across horizons (top 20 factors):")
display(icir_pivot)

## Cell 6 — ML Per Contract (LightGBM)

For each selected contract, train a separate LightGBM model:
- Features: all factors in `FACTORS`  
- Label: `ML_HORIZON`-day forward return  
- Evaluate on OOS test period: IC, directional accuracy, Sharpe

Uses `num_leaves=16` (small tree) to reduce overfitting risk on ~500 training rows.

In [ ]:
from functools import partial
from tqdm import tqdm as tqdm_nb
from vnpy.alpha.dataset.processor import process_drop_na, process_robust_zscore_norm

ml_results = []
all_signals: dict = {}  # symbol -> DataFrame[datetime, pred, ret_1d, label]

for symbol in tqdm_nb(SELECTED, desc="ML Training"):
    df_c = df.filter(pl.col("vt_symbol") == symbol)

    dataset = AlphaDataset(df_c, TRAIN_PERIOD, VALID_PERIOD, TEST_PERIOD)

    for name, expr in FACTORS.items():
        dataset.add_feature(name, expr)

    dataset.set_label(f"ts_delay(close, -{ML_HORIZON}) / close - 1")
    dataset.prepare_data(max_workers=1)

    # Drop rows with NaN labels, then normalise features
    dataset.add_processor("learn", partial(process_drop_na, names=["label"]))
    dataset.add_processor("infer", partial(process_drop_na, names=["label"]))
    dataset.add_processor("infer", process_robust_zscore_norm)
    dataset.add_processor("learn", process_robust_zscore_norm)
    dataset.process_data()

    train_rows = len(dataset.fetch_learn(Segment.TRAIN))
    if train_rows < 100:
        print(f"  WARN: {symbol} only {train_rows} training rows -- skipping ML")
        continue

    model = LgbModel(
        learning_rate=0.05,
        num_leaves=16,
        num_boost_round=500,
        early_stopping_rounds=30,
        log_evaluation_period=100,
        seed=42
    )
    model.fit(dataset)

    # Predict on test
    preds = model.predict(dataset, Segment.TEST)
    test_df = dataset.fetch_infer(Segment.TEST).sort(["datetime", "vt_symbol"]).to_pandas().dropna()

    if len(test_df) == 0:
        print(f"  WARN: {symbol} no test rows after dropna")
        continue

    test_df["pred"] = preds[:len(test_df)]
    test_df = test_df.dropna(subset=["label", "pred"])

    if len(test_df) < 20:
        print(f"  WARN: {symbol} insufficient test rows ({len(test_df)})")
        continue

    # OOS IC
    oos_ic, _ = spearmanr(test_df["pred"], test_df["label"])

    # Directional accuracy
    dir_acc = (np.sign(test_df["pred"]) == np.sign(test_df["label"])).mean()

    # Simple long/short daily Sharpe: go long when pred>0, short when pred<0
    df_raw_c = df.filter(pl.col("vt_symbol") == symbol).sort("datetime").to_pandas()
    df_raw_c["ret_1d"] = df_raw_c["close"].pct_change(1).shift(-1)
    df_raw_c["datetime"] = pd.to_datetime(df_raw_c["datetime"])
    test_df["datetime"] = pd.to_datetime(test_df["datetime"])

    merged = test_df.merge(df_raw_c[["datetime", "ret_1d"]], on="datetime", how="left").dropna(subset=["ret_1d"])

    signal_ret = np.sign(merged["pred"]) * merged["ret_1d"]
    sharpe = (signal_ret.mean() / signal_ret.std()) * np.sqrt(242) if signal_ret.std() > 0 else 0.0
    total_ret = (1 + signal_ret).prod() - 1

    ml_results.append({
        "symbol": symbol,
        "train_rows": train_rows,
        "test_rows": len(test_df),
        "OOS_IC": round(oos_ic, 4),
        "Dir_Acc": round(dir_acc, 4),
        "OOS_Sharpe": round(sharpe, 3),
        "OOS_TotalRet": round(total_ret * 100, 2),
    })

    # Store signal for detailed evaluation in Cell 9
    all_signals[symbol] = merged[["datetime", "pred", "ret_1d", "label"]].copy()

ml_df = pd.DataFrame(ml_results).sort_values("OOS_Sharpe", ascending=False)
print(f"\nML results for {len(ml_df)} contracts (sorted by OOS Sharpe):")
display(ml_df)

## Cell 7 — Summary Report

In [ ]:
# ── Best factor per contract (by |mean IC| over HORIZONS) ───────────
best_factor = (
    ic_df.groupby(["symbol", "factor"])["IC"]
    .agg(lambda x: abs(x.mean()))
    .reset_index()
    .rename(columns={"IC": "abs_mean_IC"})
    .sort_values("abs_mean_IC", ascending=False)
    .groupby("symbol")
    .first()
    .reset_index()[["symbol", "factor", "abs_mean_IC"]]
    .rename(columns={"factor": "best_factor", "abs_mean_IC": "best_factor_IC"})
)

# Best horizon per contract
best_horizon = (
    ic_df.groupby(["symbol", "horizon"])["IC"]
    .agg(lambda x: abs(x.mean()))
    .reset_index()
    .rename(columns={"IC": "abs_mean_IC"})
    .sort_values("abs_mean_IC", ascending=False)
    .groupby("symbol")
    .first()
    .reset_index()[["symbol", "horizon"]]
    .rename(columns={"horizon": "best_horizon"})
)

# Merge with ML results
summary = best_factor.merge(best_horizon, on="symbol", how="left")
if len(ml_df) > 0:
    summary = summary.merge(ml_df[["symbol", "OOS_IC", "OOS_Sharpe", "OOS_TotalRet", "Dir_Acc"]],
                            on="symbol", how="left")

summary = summary.sort_values("OOS_Sharpe", ascending=False).reset_index(drop=True)
summary.index += 1
summary.columns = ["Contract", "Best Factor", "Best Factor |IC|", "Best Horizon",
                   "OOS IC", "OOS Sharpe", "OOS Total Ret (%)", "Directional Acc"]

print("="*80)
print(" SUMMARY: Time-Series Factor Research")
print(f" Factors tested: {list(FACTORS.keys())}")
print(f" Horizons: 1-{max(HORIZONS)}d | ML horizon: {ML_HORIZON}d")
print("="*80)
display(summary.round(4))

# ── Chart: OOS Sharpe per contract ──────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

if len(ml_df) > 0:
    ml_plot = ml_df.sort_values("OOS_Sharpe")
    colors = ["#2ecc71" if v > 0 else "#e74c3c" for v in ml_plot["OOS_Sharpe"]]
    ax1.barh(ml_plot["symbol"], ml_plot["OOS_Sharpe"], color=colors)
    ax1.axvline(0, color="black", linewidth=0.8, linestyle="--")
    ax1.axvline(1.0, color="#27ae60", linewidth=1, linestyle=":", label="Sharpe=1.0")
    ax1.set_title(f"OOS Sharpe by Contract (ML_HORIZON={ML_HORIZON}d)", fontweight="bold")
    ax1.set_xlabel("Annualised Sharpe Ratio (OOS)")
    ax1.legend()

# Mean |IC| per factor (bar chart)
mean_ic_factor = (
    ic_df.groupby("factor")["IC"]
    .agg(lambda x: x.mean())
    .reset_index()
    .sort_values("IC", ascending=True)
)
colors2 = ["#2ecc71" if v > 0 else "#e74c3c" for v in mean_ic_factor["IC"]]
ax2.barh(mean_ic_factor["factor"], mean_ic_factor["IC"], color=colors2)
ax2.axvline(0, color="black", linewidth=0.8, linestyle="--")
ax2.axvline(0.05, color="#27ae60", linewidth=1, linestyle=":", label="IC=0.05")
ax2.axvline(-0.05, color="#e74c3c", linewidth=1, linestyle=":")
ax2.set_title("Mean IC per Factor (across all contracts & horizons)", fontweight="bold")
ax2.set_xlabel("Mean Spearman IC")
ax2.legend()

plt.tight_layout()
plt.savefig("./lab/futures/ts_summary.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: ts_summary.png")

# Highlight strong signals
print("\n── Contracts with OOS Sharpe > 0.5 ──")
if len(ml_df) > 0:
    strong = ml_df[ml_df["OOS_Sharpe"] > 0.5]
    if len(strong) > 0:
        display(strong)
    else:
        print("  None found — consider adding more factors or adjusting ML_HORIZON")

print("\n── Factors with |mean IC| > 0.05 ──")
strong_ic = mean_ic_factor[abs(mean_ic_factor["IC"]) > 0.05]
if len(strong_ic) > 0:
    display(strong_ic)
else:
    print("  None found — factors may be weak for this universe/period")

## Cell 8 — Deep Factor Analysis

This cell drills into **factor quality** using three lenses:

1. **IC Decay Curve** — Mean IC vs forecast horizon for the top factors. A steep decay (IC drops sharply beyond 1–2 days) indicates a short-term signal; a flat curve indicates persistent predictive power.

2. **ICIR Ranking** — `ICIR = mean(IC) / std(IC)` is the signal-to-noise ratio of a factor. A factor with mean IC = 0.04 but std(IC) = 0.01 (ICIR = 4.0) is far more reliable than one with mean IC = 0.08 but std(IC) = 0.20 (ICIR = 0.4). Threshold: |ICIR| > 0.5 = consistent, > 1.0 = strong.

3. **Rolling IC Over Time** — 60-day rolling Spearman IC for the top factor on the top contracts. Shows whether the factor's predictive power is **stable** or **regime-dependent** (fading in certain periods = edge decay signal, analogous to rolling Sharpe for strategy evaluation).

In [ ]:
TOP_FACTORS_N = 10   # how many top factors to show in IC decay chart
ROLL_WINDOW   = 60   # days for rolling IC

# ── ICIR table: signal-to-noise ratio for every factor ───────────────────────
icir_df = (
    ic_df.groupby("factor")["IC"]
    .agg(
        mean_ic=lambda x: x.mean(),
        std_ic=lambda x: x.std(),
        icir=lambda x: x.mean() / x.std() if x.std() > 0 else 0.0,
        abs_mean_ic=lambda x: abs(x.mean()),
    )
    .sort_values("icir", ascending=False)
    .reset_index()
)

top_factors_by_icir = icir_df.head(TOP_FACTORS_N)["factor"].tolist()

# ── 1. IC Decay Curve ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

ax = axes[0]
for factor in top_factors_by_icir:
    fh_ic = (
        ic_df[ic_df["factor"] == factor]
        .groupby("horizon")["IC"]
        .mean()
    )
    ax.plot(fh_ic.index, fh_ic.values, marker="o", label=factor, linewidth=1.5)

ax.axhline(0,    color="black", linewidth=0.8, linestyle="--")
ax.axhline(0.05, color="green", linewidth=0.8, linestyle=":", alpha=0.7, label="IC=±0.05")
ax.axhline(-0.05, color="green", linewidth=0.8, linestyle=":", alpha=0.7)
ax.set_xlabel("Forecast Horizon (days)")
ax.set_ylabel("Mean IC (across all selected contracts)")
ax.set_title(f"IC Decay by Horizon — Top {TOP_FACTORS_N} Factors by |ICIR|", fontweight="bold")
ax.legend(fontsize=7, ncol=2)
ax.grid(True, alpha=0.3)
ax.set_xticks(HORIZONS)

# ── 2. ICIR Bar Chart (top 20) ────────────────────────────────────────────────
ax2 = axes[1]
top20_icir = icir_df.head(20).sort_values("icir")
colors = ["#2ecc71" if v > 0 else "#e74c3c" for v in top20_icir["icir"]]
ax2.barh(top20_icir["factor"], top20_icir["icir"], color=colors)
ax2.axvline(0,    color="black", linewidth=0.8, linestyle="--")
ax2.axvline(0.5,  color="green", linewidth=1,   linestyle=":", alpha=0.7, label="|ICIR|=0.5")
ax2.axvline(-0.5, color="red",   linewidth=1,   linestyle=":", alpha=0.7)
ax2.set_xlabel("ICIR = mean(IC) / std(IC)")
ax2.set_title("Top 20 Factors by ICIR (Signal-to-Noise Ratio)", fontweight="bold")
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("./lab/futures/factor_decay_icir.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: factor_decay_icir.png")

# ── Full ICIR table ───────────────────────────────────────────────────────────
print("\nFull factor ICIR ranking:")
display(icir_df.round(4))

# ── 3. Rolling IC Over Time ───────────────────────────────────────────────────
# Best factor (by ICIR) on top-5 contracts (by OOS Sharpe)
best_factor_name = icir_df.iloc[0]["factor"]
top_roll_contracts = ml_df.head(5)["symbol"].tolist() if len(ml_df) >= 5 else SELECTED[:5]

print(f"\nComputing rolling {ROLL_WINDOW}-day IC for factor '{best_factor_name}' ...")


def rolling_spearman(factor_s: pd.Series, fwd_s: pd.Series, window: int) -> pd.Series:
    """Row-by-row rolling Spearman correlation — O(N × window), fine for ≤2000 rows."""
    aligned = pd.concat([factor_s.rename("f"), fwd_s.rename("r")], axis=1).dropna()
    ics, idx = [], []
    for i in range(len(aligned)):
        if i < window - 1:
            ics.append(np.nan)
        else:
            w = aligned.iloc[i - window + 1: i + 1]
            ic_val, _ = spearmanr(w["f"], w["r"])
            ics.append(ic_val)
        idx.append(aligned.index[i])
    return pd.Series(ics, index=idx)


fig, axes_roll = plt.subplots(len(top_roll_contracts), 1,
                               figsize=(16, 3.5 * len(top_roll_contracts)),
                               sharex=False)
if len(top_roll_contracts) == 1:
    axes_roll = [axes_roll]

for ax, symbol in zip(axes_roll, top_roll_contracts):
    df_c = df.filter(pl.col("vt_symbol") == symbol)
    ds_roll = AlphaDataset(df_c, TRAIN_PERIOD, VALID_PERIOD, TEST_PERIOD)
    ds_roll.add_feature(best_factor_name, FACTORS[best_factor_name])
    ds_roll.add_feature("fwd_1d", "ts_delay(close, -1) / close - 1")
    ds_roll.prepare_data(max_workers=1)

    raw_c = ds_roll.result_df.sort("datetime").to_pandas().set_index("datetime")
    roll_ic = rolling_spearman(raw_c[best_factor_name], raw_c["fwd_1d"], ROLL_WINDOW)

    mean_val = roll_ic.dropna().mean()
    ax.plot(roll_ic.index, roll_ic.values, color="steelblue", linewidth=1.2)
    ax.axhline(0,    color="black",  linewidth=0.8, linestyle="--")
    ax.axhline(0.1,  color="green",  linewidth=0.8, linestyle=":", alpha=0.7)
    ax.axhline(-0.1, color="red",    linewidth=0.8, linestyle=":", alpha=0.7)
    ax.fill_between(roll_ic.index, roll_ic.values, 0,
                    where=(roll_ic > 0), color="green", alpha=0.15)
    ax.fill_between(roll_ic.index, roll_ic.values, 0,
                    where=(roll_ic < 0), color="red",   alpha=0.15)
    ax.set_title(
        f"{symbol} — Rolling {ROLL_WINDOW}d IC | factor: {best_factor_name} | mean = {mean_val:.3f}",
        fontweight="bold"
    )
    ax.set_ylabel("Rolling IC")
    ax.grid(True, alpha=0.3)

plt.suptitle(
    f"Rolling IC Over Time — Factor: {best_factor_name}  (window={ROLL_WINDOW}d)",
    fontsize=13, fontweight="bold", y=1.01
)
plt.tight_layout()
plt.savefig("./lab/futures/rolling_ic.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: rolling_ic.png")

## Cell 9 — ML Signal Full Evaluation

Full backtesting-quality evaluation for each contract's ML signal, following the standard evaluation framework:

| Chart / Metric | Purpose |
|---|---|
| **Equity curve + drawdown** | Most important — shows cumulative P&L and worst loss periods |
| **Rolling Sharpe (60d)** | Detects edge decay: declining rolling Sharpe = signal fading |
| **Monthly returns heatmap** | Reveals regime dependence, seasonality, and all-red years |
| **Sharpe / Sortino / Calmar / Max DD** | Full risk-adjusted metric suite (China futures: 242 days/year, RF=2.5%) |
| **Win rate / Profit Factor / Expectancy** | Trade statistics — high win rate alone is not enough |
| **t-statistic** | Is mean return significantly > 0? Need t > 2.0 for 95% confidence |
| **Outlier sensitivity** | Remove best 3 and 5 days — robust signal stays profitable |
| **Monte Carlo (2000 simulations)** | Bootstrap P(profitable), realistic p5 max drawdown |

**Interpretation thresholds** (from `backtesting_evaluation.md`):
- Sharpe > 1.0 = acceptable, > 2.0 = strong, > 3.0 = red flag
- Calmar > 1.0 = good, > 2.0 = strong
- Monte Carlo P(profitable) > 70% = robust, < 60% = fragile edge
- Removing best 3 days must NOT flip profit→loss — if it does, the strategy is outlier-driven

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr

# ═══════════════════════════════════════════════════════════════════════════
#  Helper functions (backtesting_evaluation.md framework)
#  China futures: periods_per_year=242, risk_free=2.5%
# ═══════════════════════════════════════════════════════════════════════════
PERIODS_PER_YEAR = 242
RISK_FREE = 0.025


def _drawdown_series(returns: pd.Series) -> pd.Series:
    cum = (1 + returns).cumprod()
    return (cum / cum.expanding().max()) - 1


def _compute_metrics(returns: pd.Series) -> dict:
    rf_d = RISK_FREE / PERIODS_PER_YEAR
    excess = returns - rf_d
    dd = _drawdown_series(returns)
    max_dd = dd.min()
    n = len(returns)

    sharpe = np.sqrt(PERIODS_PER_YEAR) * excess.mean() / excess.std() if excess.std() > 0 else 0.0
    downside = returns[returns < rf_d]
    ds_std = downside.std() * np.sqrt(PERIODS_PER_YEAR) if len(downside) > 1 else np.nan
    sortino = (returns.mean() * PERIODS_PER_YEAR - RISK_FREE) / ds_std if ds_std and ds_std > 0 else 0.0
    years = n / PERIODS_PER_YEAR
    total_ret = (1 + returns).prod() - 1
    cagr = (1 + total_ret) ** (1 / years) - 1 if years > 0 else 0.0
    calmar = abs(cagr / max_dd) if max_dd != 0 else 0.0

    wins = returns[returns > 0]
    losses = returns[returns < 0]
    win_rate = len(wins) / n if n > 0 else 0.0
    avg_win = wins.mean() if len(wins) > 0 else 0.0
    avg_loss = abs(losses.mean()) if len(losses) > 0 else 0.0
    pf = wins.sum() / abs(losses.sum()) if losses.sum() != 0 else np.inf
    exp = win_rate * avg_win - (1 - win_rate) * avg_loss
    t_stat = returns.mean() / (returns.std(ddof=1) / np.sqrt(n)) if returns.std() > 0 else 0.0

    return {
        "Total Ret %":   round(total_ret * 100, 1),
        "CAGR %":        round(cagr * 100, 1),
        "Sharpe":        round(sharpe, 3),
        "Sortino":       round(sortino, 3),
        "Calmar":        round(calmar, 3),
        "Max DD %":      round(max_dd * 100, 1),
        "Win Rate %":    round(win_rate * 100, 1),
        "Profit Factor": round(pf, 3),
        "Exp %/day":     round(exp * 100, 4),
        "t-stat":        round(t_stat, 3),
        "n_days":        n,
    }


def _outlier_sensitivity(returns: pd.Series, k: int = 5) -> pd.DataFrame:
    rows, sorted_r = [], returns.sort_values()
    scenarios = [("All days", returns),
                 *[(f"Remove best {i}", sorted_r.iloc[:-i]) for i in range(1, k + 1)],
                 *[(f"Remove worst {i}", sorted_r.iloc[i:]) for i in range(1, k + 1)]]
    for label, r in scenarios:
        sh = np.sqrt(PERIODS_PER_YEAR) * r.mean() / r.std() if r.std() > 0 else 0
        rows.append({
            "scenario":      label,
            "total_ret_%":   round((1 + r).prod() * 100 - 100, 2),
            "sharpe":        round(sh, 3),
            "n_days":        len(r),
        })
    return pd.DataFrame(rows)


def _monte_carlo(returns: pd.Series, n_sim: int = 2000, seed: int = 0) -> dict:
    rng = np.random.default_rng(seed)
    trades = returns.values
    n = len(trades)
    final_rets, max_dds = [], []
    for _ in range(n_sim):
        sim = rng.choice(trades, size=n, replace=True)
        eq = np.cumprod(1 + sim)
        final_rets.append(eq[-1] - 1)
        rm = np.maximum.accumulate(eq)
        max_dds.append(((eq - rm) / rm).min())
    fr, md = np.array(final_rets), np.array(max_dds)
    return {
        "P(Profitable)": round(float((fr > 0).mean()), 3),
        "p5 Ret %":      round(float(np.percentile(fr, 5) * 100), 1),
        "p50 Ret %":     round(float(np.percentile(fr, 50) * 100), 1),
        "p95 Ret %":     round(float(np.percentile(fr, 95) * 100), 1),
        "p5 MaxDD %":    round(float(np.percentile(md, 5) * 100), 1),
        "Actual Ret %":  round(float((1 + returns).prod() - 1) * 100, 1),
    }


# ═══════════════════════════════════════════════════════════════════════════
#  Build per-contract returns Series from stored all_signals
# ═══════════════════════════════════════════════════════════════════════════
if not all_signals:
    print("No signals stored — re-run Cell 6 first.")
else:
    def _signal_returns(symbol: str) -> pd.Series:
        sig = all_signals[symbol].dropna(subset=["ret_1d", "pred"])
        sr = pd.Series(
            np.sign(sig["pred"].values) * sig["ret_1d"].values,
            index=pd.to_datetime(sig["datetime"].values),
            name=symbol,
        ).sort_index()
        return sr

    eval_symbols = list(all_signals.keys())

    # ── Full metrics table ────────────────────────────────────────────────
    print("=" * 70)
    print(" DETAILED SIGNAL EVALUATION (per contract)")
    print(f" China futures: {PERIODS_PER_YEAR} days/year, RF={RISK_FREE*100:.1f}%")
    print("=" * 70)

    metrics_rows = {sym: _compute_metrics(_signal_returns(sym)) for sym in eval_symbols}
    metrics_tbl = pd.DataFrame(metrics_rows).T.sort_values("Sharpe", ascending=False)
    display(metrics_tbl)

    # ── Equity curves + underwater + rolling Sharpe (one row per contract) ─
    n = len(eval_symbols)
    fig, axes_ec = plt.subplots(n, 2, figsize=(16, 3.5 * n), squeeze=False)

    for row_i, symbol in enumerate(eval_symbols):
        sr = _signal_returns(symbol)
        cum = (1 + sr).cumprod()
        dd = _drawdown_series(sr)

        # Left: equity + underwater overlay
        ax_eq = axes_ec[row_i][0]
        ax_eq.plot(cum.index, cum.values, color="steelblue", linewidth=1.5, label="Equity")
        ax_eq.set_title(f"{symbol} — OOS Equity Curve", fontweight="bold")
        ax_eq.set_ylabel("Cumulative Return")
        ax_eq.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.2f}x"))
        ax_eq.grid(True, alpha=0.3)
        ax_dd = ax_eq.twinx()
        ax_dd.fill_between(dd.index, dd.values, 0, color="crimson", alpha=0.25)
        ax_dd.plot(dd.index, dd.values, color="crimson", linewidth=0.8)
        ax_dd.set_ylabel("Drawdown", color="crimson")
        ax_dd.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x*100:.0f}%"))
        ax_dd.tick_params(axis="y", labelcolor="crimson")

        # Right: rolling Sharpe
        roll_w = min(60, max(20, len(sr) // 4))
        exc = sr - RISK_FREE / PERIODS_PER_YEAR
        roll_sh = (exc.rolling(roll_w).mean() / exc.rolling(roll_w).std()) * np.sqrt(PERIODS_PER_YEAR)
        ax_sh = axes_ec[row_i][1]
        ax_sh.plot(roll_sh.index, roll_sh.values, color="steelblue", linewidth=1.2)
        ax_sh.axhline(0,   color="black", linewidth=0.8, linestyle="--")
        ax_sh.axhline(1.0, color="green", linewidth=0.8, linestyle=":", alpha=0.7, label="Sharpe=1.0")
        ax_sh.fill_between(roll_sh.index, roll_sh.values, 0,
                            where=(roll_sh > 0), color="green", alpha=0.15)
        ax_sh.fill_between(roll_sh.index, roll_sh.values, 0,
                            where=(roll_sh < 0), color="red",   alpha=0.15)
        ax_sh.set_title(f"{symbol} — Rolling {roll_w}d Sharpe", fontweight="bold")
        ax_sh.set_ylabel("Rolling Sharpe")
        ax_sh.legend(fontsize=8)
        ax_sh.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig("./lab/futures/signal_equity_sharpe.png", dpi=100, bbox_inches="tight")
    plt.show()
    print("Saved: signal_equity_sharpe.png")

    # ── Monthly returns heatmaps (top-3 by OOS Sharpe) ───────────────────
    top3 = metrics_tbl.head(3).index.tolist()
    fig2, axes_mh = plt.subplots(1, len(top3), figsize=(7 * len(top3), 5), squeeze=False)

    for ax_m, symbol in zip(axes_mh[0], top3):
        sr = _signal_returns(symbol)
        monthly = sr.resample("ME").apply(lambda x: (1 + x).prod() - 1) * 100
        df_m = pd.DataFrame({
            "year":   monthly.index.year,
            "month":  monthly.index.month,
            "return": monthly.values,
        })
        pivot = df_m.pivot(index="year", columns="month", values="return")
        month_labels = ["Jan","Feb","Mar","Apr","May","Jun",
                        "Jul","Aug","Sep","Oct","Nov","Dec"]
        pivot.columns = [month_labels[m - 1] for m in pivot.columns]
        sns.heatmap(pivot, annot=True, fmt=".1f", center=0, cmap="RdYlGn",
                    linewidths=0.5, ax=ax_m, annot_kws={"size": 8})
        ax_m.set_title(f"{symbol}\nMonthly Returns (%)", fontweight="bold")

    plt.tight_layout()
    plt.savefig("./lab/futures/monthly_heatmaps.png", dpi=100, bbox_inches="tight")
    plt.show()
    print("Saved: monthly_heatmaps.png")

    # ── Outlier sensitivity ───────────────────────────────────────────────
    print("\n--- Outlier Sensitivity (remove best / worst N days) ---")
    print("Rule: robust signal stays profitable after removing best 3 days.")
    for symbol in top3:
        out_df = _outlier_sensitivity(_signal_returns(symbol), k=5)
        print(f"\n  {symbol}:")
        display(out_df)

    # ── Monte Carlo ───────────────────────────────────────────────────────
    print("\n--- Monte Carlo (2000 bootstrap simulations) ---")
    print("P(profitable) > 70% = robust | < 60% = fragile edge")
    mc_rows = {}
    for symbol in eval_symbols:
        mc_rows[symbol] = _monte_carlo(_signal_returns(symbol), n_sim=2000)
    mc_tbl = pd.DataFrame(mc_rows).T.sort_values("P(Profitable)", ascending=False)
    display(mc_tbl)